### Daten erstellen und customclassifier fitten

In [1]:
from utils import create_train_test_data, create_new_dataset_with_ipcw_weights
from class_DecisionTreeBaggingClassifier import DecisionTreeBaggingClassifier
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter

n = 20
seed = 42
tau = 32
B_RF = int(0.7 * n * 2)
B_first_level = 20

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': seed, 'tau': tau}


df_train, df_test\
    , n_events_after_cut_train, portion_censored_after_cut_train\
    , n_events_after_cut_test, portion_censored_after_cut_test = create_train_test_data(params=params_data_creation)

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  seed,
                'weighted_bootstrapping':     True,  }

X_pred_point = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})
clf = DecisionTreeBaggingClassifier(params_rf)

clf.fit(
            X=df_train.drop(
                ["time", "event", "weights_ipcw", "survived"], axis=1
            ).values,
            y=df_train["survived"].values,
            sample_weights=df_train["weights_ipcw"].values,)

